# Ultradian/Circadian Screening Pipeline (Morph2REP / Envision 2025v3.3)

This notebook is a **drop-in screening pipeline** intended to test for evidence of a **~4-day cyclic signal** in individual mice using high-resolution behavior, while also quantifying:

- **Per-animal** evidence (per-mouse GMM; primary unit of inference)
- **Cage-to-cage** differences (summary + comparison)
- **Replicate** differences (Rep1 vs Rep2 summary + comparison)
- **Event-day sensitivity** (event-only vs event±1 vs “exclude events” variants)

## Why `ANCHOR_HOUR = 6` matters (behavioral day definition)

The Morph2REP technical documentation specifies a **light cycle boundary at 6:00 AM / 6:00 PM (EST)**.  
To avoid splitting light/dark cycles across “calendar days”, we define a **behavioral day** as:

> **06:00 → 06:00**

This is controlled by `ANCHOR_HOUR = 6`, meaning:
- timestamps from **00:00–05:59** belong to the **previous** behavioral day
- the 6:00 AM light-cycle transition is the day boundary

This typically improves daily feature stability for circadian/estrous analyses.

---

## Data source used here (first test)
We start with **`animal_bouts.parquet`** and treat **locomotion** as a 1-minute binary occupancy signal.
- This is closest to your prior “positive control” method style (bout-to-minute signal → wavelets → daily features → per-mouse GMM).

Later, you can switch adapters to use other tables (e.g., `animal_aggs_short_id.parquet` with a chosen metric name).

In [1]:
# =============================================================================
# 0) Imports
# =============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import duckdb

from scipy.stats import mannwhitneyu
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA

import pywt
import warnings
warnings.filterwarnings("ignore")

plt.rcParams["figure.dpi"] = 140
plt.rcParams["font.size"] = 10

print("Imports ready.")

Imports ready.


---
## 1) USER CONFIG

### What you change between datasets
In most cases you only change:
- the **cage lists** and date ranges per replicate
- the **event timestamps** (if you want event sensitivity)
- optionally `ANCHOR_HOUR` (but for Morph2REP keep it at 6)

This config block is designed to be the *only* part you edit for a new run.

In [2]:
# =============================================================================
# USER CONFIG (edit these)
# =============================================================================

# S3 base path (public)
S3_BASE = "s3://jax-envision-public-data/study_1001/2025v3.3/tabular"

# Adapter (this notebook implements BOUTS_LOCO_BIN end-to-end)
DATA_ADAPTER = "BOUTS_LOCO_BIN"

# Behavioral day anchor
# 0  => 00:00–00:00
# 6  => 06:00–06:00  (recommended; matches Morph2REP light-cycle boundary in docs)
ANCHOR_HOUR = 6

# Minute grid constant
MINUTES_PER_DAY = 1440

# Envision bout state for locomotion
LOCO_STATE_NAME = "animal_bouts.locomotion"

# ---- Study layout (start with Vehicle cages; add dose cages later if desired)
GROUPS = {
    "vehicle": {
        "Rep1": {
            "cages": [4918, 4922, 4923],
            "analysis_start": "2025-01-07",
            "analysis_end":   "2025-01-22",
        },
        "Rep2": {
            "cages": [4928, 4929, 4934],
            "analysis_start": "2025-01-22",
            "analysis_end":   "2025-02-04",
        },
    },
    # "dose5": {...},
    # "dose25": {...},
}

# ---- Event timestamps from Morph2REP documentation (EST)
# Leave empty to disable event analyses.
EVENTS = {
    "Rep1": [
        ("dose_1",      pd.Timestamp("2025-01-14 06:00:00")),
        ("cage_change", pd.Timestamp("2025-01-15 12:00:00")),
        ("dose_2",      pd.Timestamp("2025-01-17 17:00:00")),
    ],
    "Rep2": [
        ("dose_1",      pd.Timestamp("2025-01-28 17:00:00")),
        ("cage_change", pd.Timestamp("2025-01-29 12:00:00")),
        ("dose_2",      pd.Timestamp("2025-01-31 06:00:00")),
    ],
}

# Event masking mode for sensitivity checks:
#   "event_only"     -> just the event day
#   "event_or_next"  -> event day plus the following day (event±1 forward)
EVENT_MASK_MODE = "event_or_next"

# ---- QC settings (coverage-based)
MINUTES_FLOOR = 600      # keep days with >= this many "present" minutes
RATIO_FLOOR   = 0.30     # coverage_ratio >= RATIO_FLOOR

# ---- Wavelet settings (minute-resolution signal)
DT_SECONDS = 60.0
WAVELET = "cmor1.5-1.0"
N_SCALES = 64
PERIOD_HOURS_MIN = 1.0
PERIOD_HOURS_MAX = 39.0
ULTRADIAN_HOURS = (1.0, 3.0)
CIRCADIAN_HOURS = (23.0, 25.0)

# ---- Per-mouse analysis stability
MIN_DAYS_PER_MOUSE = 8

# ---- Cyclicity test
LAG_DAYS = 4
N_PERM = 3000
RANDOM_SEED = 0

print("Config loaded. Groups:", list(GROUPS.keys()))

Config loaded. Groups: ['vehicle']


---
## 2) DuckDB S3 Loader (memory-safe)

We use DuckDB `read_parquet()` on S3 and only load needed columns.

In [3]:
def make_duckdb_conn():
    conn = duckdb.connect()
    conn.execute("INSTALL httpfs; LOAD httpfs;")
    conn.execute("SET s3_region='us-east-1';")
    return conn

def load_parquet_s3(conn, cage_id: int, date: str, table_name: str,
                    columns=None, where=None) -> pd.DataFrame:
    cols_sql = "*" if (not columns) else ", ".join(columns)
    where_sql = "" if (not where) else f" WHERE {where}"
    path = f"{S3_BASE}/cage_id={cage_id}/date={date}/{table_name}"
    try:
        df = conn.execute(f"SELECT {cols_sql} FROM read_parquet('{path}'){where_sql}").fetchdf()
        if df is None or len(df) == 0:
            return pd.DataFrame()
        df["cage_id"] = int(cage_id)
        df["date"] = str(date)
        return df
    except Exception:
        return pd.DataFrame()

def daterange(start: str, end: str):
    return [d.strftime("%Y-%m-%d") for d in pd.date_range(start, end, freq="D")]

---
## 3) Core utilities

In [4]:
def make_behavior_day_grid(analysis_start: str, analysis_end: str, anchor_hour: int):
    """Returns (t0, n_days, day_dates)."""
    start = pd.Timestamp(analysis_start)
    end = pd.Timestamp(analysis_end)
    t0 = pd.Timestamp(f"{analysis_start} {anchor_hour:02d}:00:00")
    n_days = (end.date() - start.date()).days + 1
    day_dates = [(t0 + pd.Timedelta(days=i)).strftime("%Y-%m-%d") for i in range(n_days)]
    return t0, int(n_days), day_dates

def ensure_end_time(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "end_time" in df.columns:
        df["end_time"] = pd.to_datetime(df["end_time"])
    else:
        if "bout_length_seconds" in df.columns:
            df["end_time"] = pd.to_datetime(df["start_time"]) + pd.to_timedelta(df["bout_length_seconds"], unit="s")
        else:
            raise ValueError("No end_time and no bout_length_seconds available.")
    df["start_time"] = pd.to_datetime(df["start_time"])
    return df

def compute_minutes_with_any_bout(df_bouts_allstates: pd.DataFrame, analysis_start: str, analysis_end: str, anchor_hour: int) -> pd.DataFrame:
    """Per animal per behavioral-day union minutes covered by ANY bout."""
    t0, n_days, day_dates = make_behavior_day_grid(analysis_start, analysis_end, anchor_hour)
    n_minutes = int(n_days * MINUTES_PER_DAY)
    t1 = t0 + pd.Timedelta(minutes=n_minutes)

    df = ensure_end_time(df_bouts_allstates)
    df = df[(df["end_time"] > t0) & (df["start_time"] < t1)].copy()
    if df.empty:
        return pd.DataFrame(columns=["animal_id","day","minutes_with_any_bout","date"])

    s = (df["start_time"].clip(lower=t0) - t0).dt.total_seconds().to_numpy() / 60.0
    e = (df["end_time"].clip(upper=t1)   - t0).dt.total_seconds().to_numpy() / 60.0

    start_bin = np.clip(np.floor(s).astype(np.int32), 0, n_minutes)
    end_bin   = np.clip(np.ceil(e).astype(np.int32),  0, n_minutes)
    valid = end_bin > start_bin
    if not np.any(valid):
        return pd.DataFrame(columns=["animal_id","day","minutes_with_any_bout","date"])

    df2 = pd.DataFrame({
        "animal_id": df.loc[valid, "animal_id"].astype(int).to_numpy(),
        "start_bin": start_bin[valid],
        "end_bin": end_bin[valid],
    })

    pieces = []
    for aid, g in df2.groupby("animal_id", sort=False):
        for s0, e0 in g[["start_bin","end_bin"]].to_numpy():
            d_start = int(s0) // MINUTES_PER_DAY + 1
            d_end   = (int(e0) - 1) // MINUTES_PER_DAY + 1
            for day in range(d_start, d_end + 1):
                day0 = (day - 1) * MINUTES_PER_DAY
                ls = max(int(s0), day0) - day0
                le = min(int(e0), day0 + MINUTES_PER_DAY) - day0
                if le > ls:
                    pieces.append((int(aid), int(day), int(ls), int(le)))

    if not pieces:
        return pd.DataFrame(columns=["animal_id","day","minutes_with_any_bout","date"])

    tmp = pd.DataFrame(pieces, columns=["animal_id","day","ls","le"])

    results = []
    for (aid, day), gg in tmp.groupby(["animal_id","day"], sort=False):
        intervals = gg[["ls","le"]].to_numpy()
        intervals = intervals[np.argsort(intervals[:, 0])]
        covered = 0
        cur_s, cur_e = intervals[0]
        for s2, e2 in intervals[1:]:
            if s2 <= cur_e:
                cur_e = max(cur_e, e2)
            else:
                covered += (cur_e - cur_s)
                cur_s, cur_e = s2, e2
        covered += (cur_e - cur_s)
        results.append((aid, day, int(covered), day_dates[day-1]))

    return pd.DataFrame(results, columns=["animal_id","day","minutes_with_any_bout","date"])

def apply_qc(cov: pd.DataFrame) -> pd.DataFrame:
    cov = cov.copy()
    med = cov.groupby("animal_id")["minutes_with_any_bout"].transform("median")
    cov["coverage_ratio"] = (cov["minutes_with_any_bout"] / med).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    cov["qc_keep"] = (cov["minutes_with_any_bout"] >= MINUTES_FLOOR) & (cov["coverage_ratio"] >= RATIO_FLOOR)
    return cov

def bouts_to_minute_occupancy(df_bouts: pd.DataFrame, analysis_start: str, analysis_end: str, anchor_hour: int):
    """Returns (occ, t0, n_minutes) where occ is 0/1 per minute."""
    t0, n_days, _ = make_behavior_day_grid(analysis_start, analysis_end, anchor_hour)
    n_minutes = int(n_days * MINUTES_PER_DAY)
    t1 = t0 + pd.Timedelta(minutes=n_minutes)

    df = ensure_end_time(df_bouts)
    df = df[(df["end_time"] > t0) & (df["start_time"] < t1)].copy()
    if df.empty:
        return np.zeros(n_minutes, dtype=np.float32), t0, n_minutes

    s = (df["start_time"].clip(lower=t0) - t0).dt.total_seconds().to_numpy() / 60.0
    e = (df["end_time"].clip(upper=t1)   - t0).dt.total_seconds().to_numpy() / 60.0

    start_bin = np.clip(np.floor(s).astype(np.int32), 0, n_minutes)
    end_bin   = np.clip(np.ceil(e).astype(np.int32),  0, n_minutes)
    valid = end_bin > start_bin
    if not np.any(valid):
        return np.zeros(n_minutes, dtype=np.float32), t0, n_minutes

    diff = np.zeros(n_minutes + 1, dtype=np.int32)
    np.add.at(diff, start_bin[valid], 1)
    np.add.at(diff, end_bin[valid], -1)
    occ = np.cumsum(diff[:-1])
    return (occ > 0).astype(np.float32), t0, n_minutes

In [5]:
def build_periods_hours(n_scales: int, pmin_h: float, pmax_h: float) -> np.ndarray:
    return np.logspace(np.log10(pmin_h), np.log10(pmax_h), n_scales)

def cwt_power(signal_1min: np.ndarray, periods_hours: np.ndarray):
    dt = DT_SECONDS
    periods_seconds = periods_hours * 3600.0
    scales = periods_seconds / dt
    coeffs, _freqs = pywt.cwt(signal_1min.astype(float), scales, WAVELET, sampling_period=dt)
    power = np.abs(coeffs) ** 2
    return power, periods_hours

def band_max_ts(power: np.ndarray, periods_hours: np.ndarray, band):
    lo, hi = band
    mask = (periods_hours >= lo) & (periods_hours <= hi)
    if not np.any(mask):
        return np.zeros(power.shape[1], dtype=float)
    return np.max(power[mask, :], axis=0)

def daily_mean(ts_minute: np.ndarray, n_days: int) -> np.ndarray:
    out = np.full(n_days, np.nan, dtype=float)
    for d in range(1, n_days + 1):
        s = (d - 1) * MINUTES_PER_DAY
        e = d * MINUTES_PER_DAY
        if e <= len(ts_minute):
            out[d-1] = np.nanmean(ts_minute[s:e])
    return out

def robust_z(x: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med))
    return (x - med) / (mad + eps)

In [6]:
def events_to_day_flags(analysis_start: str, analysis_end: str, anchor_hour: int, events):
    t0, n_days, day_dates = make_behavior_day_grid(analysis_start, analysis_end, anchor_hour)

    event_days = set()
    for _name, ts in events:
        day = int(np.floor(((ts - t0) / pd.Timedelta(days=1))) + 1)
        if 1 <= day <= n_days:
            event_days.add(day)

    event_or_next = set(event_days)
    if EVENT_MASK_MODE == "event_or_next":
        for d in list(event_days):
            if d + 1 <= n_days:
                event_or_next.add(d + 1)

    rows = []
    for day in range(1, n_days + 1):
        rows.append({
            "day": day,
            "date": day_dates[day-1],
            "is_event": int(day in event_days),
            "is_event_or_next": int(day in event_or_next),
        })
    return pd.DataFrame(rows)

---
## 4) Data adapter: `animal_bouts.parquet` → daily wavelet features

In [7]:
BOUTS_COLS = ["animal_id","start_time","end_time","state_name","bout_length_seconds"]

def build_features_from_bouts(conn, group: str, rep: str, cage_id: int,
                             analysis_start: str, analysis_end: str,
                             anchor_hour: int, events):
    # Grid + event flags
    t0, n_days, day_dates = make_behavior_day_grid(analysis_start, analysis_end, anchor_hour)
    ev_df = events_to_day_flags(analysis_start, analysis_end, anchor_hour, events)

    # ------------------------------------------------------------
    # FAST LOAD: one parquet scan per cage using hive partitioning
    # ------------------------------------------------------------
    path = f"{S3_BASE}/cage_id={int(cage_id)}/date=*/animal_bouts.parquet"
    cols_sql = ", ".join(BOUTS_COLS)

    # DuckDB can read hive partitions if we enable hive_partitioning=1
    # and then we can filter on the partition column `date`.
    q = f"""
        SELECT {cols_sql}, date
        FROM read_parquet('{path}', hive_partitioning=1)
        WHERE date BETWEEN '{analysis_start}' AND '{analysis_end}'
    """

    try:
        df_all = conn.execute(q).fetchdf()
    except Exception as e:
        print(f"[WARN] cage={cage_id}: failed to read bouts via wildcard. Error: {e}")
        return pd.DataFrame()

    if df_all.empty:
        return pd.DataFrame()

    # Basic cleaning
    df_all = df_all[df_all["animal_id"].astype(int) != 0].copy()
    df_all["start_time"] = pd.to_datetime(df_all["start_time"])
    df_all["end_time"]   = pd.to_datetime(df_all["end_time"])

    # ------------------------------------------------------------
    # Coverage + QC (uses ALL bouts)
    # ------------------------------------------------------------
    cov = compute_minutes_with_any_bout(df_all, analysis_start, analysis_end, anchor_hour)
    cov = apply_qc(cov)
    cov["group"] = group
    cov["replicate"] = rep
    cov["cage_id"] = int(cage_id)

    # ------------------------------------------------------------
    # Wavelet features (uses locomotion bouts only) — no extra reads
    # ------------------------------------------------------------
    df_loco = df_all[df_all["state_name"] == LOCO_STATE_NAME].copy()

    rows = []
    periods_hours = build_periods_hours(N_SCALES, PERIOD_HOURS_MIN, PERIOD_HOURS_MAX)

    if not df_loco.empty:
        for aid, g in df_loco.groupby("animal_id", sort=False):
            occ, _t0, _nmin = bouts_to_minute_occupancy(g, analysis_start, analysis_end, anchor_hour)
            power, periods_h = cwt_power(occ, periods_hours)

            U_ts = band_max_ts(power, periods_h, ULTRADIAN_HOURS)
            C_ts = band_max_ts(power, periods_h, CIRCADIAN_HOURS)

            U_day = daily_mean(U_ts, n_days)
            C_day = daily_mean(C_ts, n_days)
            log_ratio = np.log((U_day + 1e-6) / (C_day + 1e-6))

            for day in range(1, n_days + 1):
                rows.append({
                    "group": group,
                    "replicate": rep,
                    "cage_id": int(cage_id),
                    "animal_id": int(aid),
                    "day": int(day),
                    "date": day_dates[day-1],
                    "U_1_3h": float(U_day[day-1]),
                    "C_23_25h": float(C_day[day-1]),
                    "log_U_over_C": float(log_ratio[day-1]),
                })

    feat = pd.DataFrame(rows)

    # If no locomotion rows, still return the cov frame with NaNs for wavelet features
    if feat.empty:
        feat = cov[["group","replicate","cage_id","animal_id","day","date"]].copy()
        feat["U_1_3h"] = np.nan
        feat["C_23_25h"] = np.nan
        feat["log_U_over_C"] = np.nan

    # Join coverage + QC + events
    feat = feat.merge(
        cov[["group","replicate","cage_id","animal_id","day","date","minutes_with_any_bout","coverage_ratio","qc_keep"]],
        on=["group","replicate","cage_id","animal_id","day","date"],
        how="left"
    )
    feat = feat.merge(ev_df, on=["day","date"], how="left")

    return feat

---
## 5) Per-mouse GMM + cyclicity utilities

In [8]:
def autocorr_lag(series_by_day: pd.Series, lag: int) -> float:
    s = series_by_day.sort_index()
    s2 = s.shift(-lag)
    mask = s.notna() & s2.notna()
    if mask.sum() < 3:
        return np.nan
    x0 = s[mask].astype(float).values
    x1 = s2[mask].astype(float).values
    if np.nanstd(x0) < 1e-10 or np.nanstd(x1) < 1e-10:
        return np.nan
    return float(np.corrcoef(x0, x1)[0,1])

def circular_shift(vals: np.ndarray, k: int) -> np.ndarray:
    n = len(vals)
    if n == 0:
        return vals
    k = k % n
    if k == 0:
        return vals
    return np.concatenate([vals[-k:], vals[:-k]])

def permutation_null_autocorr(series: pd.Series, lag: int, n_perm: int, rng: np.random.Generator) -> np.ndarray:
    vals = series.values
    n = len(vals)
    out = np.full(n_perm, np.nan, dtype=float)
    for i in range(n_perm):
        k = int(rng.integers(0, n)) if n > 0 else 0
        shifted = circular_shift(vals, k)
        out[i] = autocorr_lag(pd.Series(shifted, index=series.index), lag)
    return out

def per_mouse_gmm(sub: pd.DataFrame) -> pd.DataFrame:
    g = sub.sort_values("day").copy()
    if len(g) < MIN_DAYS_PER_MOUSE:
        g["p_lowU"] = np.nan
        return g

    g["z_U_1_3h"] = robust_z(g["U_1_3h"].values)
    g["z_log_U_over_C"] = robust_z(g["log_U_over_C"].values)

    X = g[["z_U_1_3h","z_log_U_over_C"]].values
    if np.any(~np.isfinite(X)):
        g["p_lowU"] = np.nan
        return g

    gmm = GaussianMixture(n_components=2, covariance_type="full", reg_covar=1e-6, random_state=0)
    gmm.fit(X)
    post = gmm.predict_proba(X)
    hard = gmm.predict(X)

    mean_zU = [g.loc[hard==k, "z_U_1_3h"].mean() for k in [0,1]]
    low_comp = int(np.argmin(mean_zU))
    g["p_lowU"] = post[:, low_comp]

    g["gmm_low_comp"] = low_comp
    g["gmm_mean_zU_comp0"] = float(mean_zU[0])
    g["gmm_mean_zU_comp1"] = float(mean_zU[1])
    g["gmm_weight0"] = float(gmm.weights_[0])
    g["gmm_weight1"] = float(gmm.weights_[1])
    return g

def run_mouse_level_pipeline(feat: pd.DataFrame) -> pd.DataFrame:
    out = []
    for (grp, rep, cage, aid), g in feat.groupby(["group","replicate","cage_id","animal_id"], sort=False):
        g2 = g[g["qc_keep"].fillna(False)].copy()
        if len(g2) == 0:
            continue
        out.append(per_mouse_gmm(g2))
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()

def summarize_mouse(g: pd.DataFrame, lag: int, n_perm: int, seed: int = 0) -> dict:
    g = g.sort_values("day").copy()
    dmin, dmax = int(g["day"].min()), int(g["day"].max())
    idx = pd.Index(range(dmin, dmax+1), name="day")
    s = g.set_index("day")["p_lowU"].reindex(idx)

    obs = autocorr_lag(s, lag)
    rng = np.random.default_rng(seed)
    null = permutation_null_autocorr(s, lag=lag, n_perm=n_perm, rng=rng)
    null_f = null[np.isfinite(null)]
    p_mc = float((np.sum(null_f >= obs) + 1) / (len(null_f) + 1)) if (np.isfinite(obs) and len(null_f)>0) else np.nan

    ev = g.loc[g["is_event_or_next"]==1, "p_lowU"].values
    ne = g.loc[g["is_event_or_next"]==0, "p_lowU"].values
    if len(ev)>0 and len(ne)>0 and np.isfinite(ev).any() and np.isfinite(ne).any():
        p_mwu = float(mannwhitneyu(ev, ne, alternative="greater").pvalue)
        mean_ev, mean_ne = float(np.nanmean(ev)), float(np.nanmean(ne))
    else:
        p_mwu, mean_ev, mean_ne = np.nan, np.nan, np.nan

    return {
        "n_days_qc": int(len(g)),
        "obs_autocorr_lag": float(obs) if np.isfinite(obs) else np.nan,
        "mc_pvalue": p_mc,
        "mean_p_event": mean_ev,
        "mean_p_non_event": mean_ne,
        "mwu_p_event_greater": p_mwu,
        "frac_p_gt_0.9": float((g["p_lowU"] > 0.9).mean()) if len(g)>0 else np.nan,
        "frac_p_lt_0.1": float((g["p_lowU"] < 0.1).mean()) if len(g)>0 else np.nan,
    }

def build_mouse_summary_table(mouse_df: pd.DataFrame, lag: int, n_perm: int) -> pd.DataFrame:
    rows = []
    for (grp, rep, cage, aid), g in mouse_df.groupby(["group","replicate","cage_id","animal_id"], sort=False):
        s = summarize_mouse(g, lag=lag, n_perm=n_perm, seed=RANDOM_SEED)
        rows.append({"group": grp, "replicate": rep, "cage_id": int(cage), "animal_id": int(aid), **s})
    return pd.DataFrame(rows)

def agg_level(df: pd.DataFrame, level_cols):
    if len(df) == 0:
        return pd.DataFrame()
    agg = df.groupby(level_cols).agg(
        n_mice=("animal_id","nunique"),
        mean_autocorr=("obs_autocorr_lag","mean"),
        median_autocorr=("obs_autocorr_lag","median"),
        frac_mice_mc_p_lt_0_05=("mc_pvalue", lambda x: float(np.mean(x < 0.05))),
        mean_event=("mean_p_event", "mean"),
        mean_non_event=("mean_p_non_event", "mean"),
        mean_mwu_p=("mwu_p_event_greater","mean"),
        frac_mice_event_p_lt_0_05=("mwu_p_event_greater", lambda x: float(np.mean(x < 0.05))),
        frac_high_p=("frac_p_gt_0.9","mean"),
    ).reset_index()
    agg["mean_event_minus_non"] = agg["mean_event"] - agg["mean_non_event"]
    return agg

def rep_vs_rep_tests(mouse_summary: pd.DataFrame, group: str) -> pd.DataFrame:
    sub = mouse_summary[mouse_summary["group"]==group].copy()
    reps = sorted(sub["replicate"].unique())
    if len(reps) < 2:
        return pd.DataFrame()
    r1, r2 = reps[0], reps[1]

    out = []
    for metric in ["obs_autocorr_lag","mean_p_event","mean_p_non_event","mean_event_minus_non"]:
        x = sub.loc[sub["replicate"]==r1, metric].astype(float).values
        y = sub.loc[sub["replicate"]==r2, metric].astype(float).values
        x = x[np.isfinite(x)]
        y = y[np.isfinite(y)]
        if len(x) >= 3 and len(y) >= 3:
            p = float(mannwhitneyu(x, y, alternative="two-sided").pvalue)
            out.append({
                "group": group,
                "metric": metric,
                "rep1": r1, "rep2": r2,
                "rep1_mean": float(np.mean(x)),
                "rep2_mean": float(np.mean(y)),
                "mwu_p_two_sided": p,
            })
    return pd.DataFrame(out)

def recompute_autocorr_masking_events(mouse_df: pd.DataFrame, mask_mode: str = "event_only") -> pd.DataFrame:
    rows = []
    for (grp, rep, cage, aid), g in mouse_df.groupby(["group","replicate","cage_id","animal_id"], sort=False):
        g = g.sort_values("day").copy()
        dmin, dmax = int(g["day"].min()), int(g["day"].max())
        idx = pd.Index(range(dmin, dmax+1), name="day")
        s = g.set_index("day")["p_lowU"].reindex(idx)

        if mask_mode == "event_only":
            mask_days = set(g.loc[g["is_event"]==1, "day"].astype(int).tolist())
        else:
            mask_days = set(g.loc[g["is_event_or_next"]==1, "day"].astype(int).tolist())

        for d in mask_days:
            if d in s.index:
                s.loc[d] = np.nan

        obs = autocorr_lag(s, LAG_DAYS)
        rows.append({
            "group": grp, "replicate": rep, "cage_id": int(cage), "animal_id": int(aid),
            "obs_autocorr_lag_masked": float(obs) if np.isfinite(obs) else np.nan,
            "mask_mode": mask_mode,
        })
    return pd.DataFrame(rows)

---
## 6) Run: Load → QC → Features → Per-mouse GMM → Summaries

In [ ]:
conn = make_duckdb_conn()

all_feat = []
for group, reps in GROUPS.items():
    for rep, cfg in reps.items():
        events = EVENTS.get(rep, [])
        for cage_id in cfg["cages"]:
            feat_c = build_features_from_bouts(
                conn,
                group=group,
                rep=rep,
                cage_id=cage_id,
                analysis_start=cfg["analysis_start"],
                analysis_end=cfg["analysis_end"],
                anchor_hour=ANCHOR_HOUR,
                events=events,
            )
            if len(feat_c) > 0:
                all_feat.append(feat_c)

feat = pd.concat(all_feat, ignore_index=True) if all_feat else pd.DataFrame()
print("Feature rows:", len(feat))
feat.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

### 6.1 QC overview plots (report-friendly)

In [ ]:
if len(feat) > 0:
    base = feat.drop_duplicates(["group","replicate","cage_id","animal_id","day"]).copy()
    for (group, rep), g in base.groupby(["group","replicate"]):
        plt.figure(figsize=(7,4))
        plt.hist(g["minutes_with_any_bout"].dropna().values, bins=30)
        plt.axvline(MINUTES_FLOOR, linewidth=3)
        plt.title(f"{group} {rep}: minutes_with_any_bout (QC floor={MINUTES_FLOOR})")
        plt.xlabel("minutes with any bout"); plt.ylabel("count (mouse-days)")
        plt.tight_layout(); plt.show()

        plt.figure(figsize=(7,4))
        plt.hist(g["coverage_ratio"].dropna().values, bins=30)
        plt.axvline(RATIO_FLOOR, linewidth=3)
        plt.title(f"{group} {rep}: coverage_ratio (QC floor={RATIO_FLOOR})")
        plt.xlabel("coverage_ratio vs mouse median"); plt.ylabel("count (mouse-days)")
        plt.tight_layout(); plt.show()

### 6.2 Fit per-mouse GMM and compute `p_lowU`

In [ ]:
mouse_df = run_mouse_level_pipeline(feat)
print("Mouse-level rows (QC-kept):", len(mouse_df))
mouse_df.head()

### 6.3 PCA diagnostic plots (rep-level, report-friendly)

In [ ]:
def plot_rep_pca(mouse_df: pd.DataFrame, group: str, rep: str):
    sub = mouse_df[(mouse_df["group"]==group) & (mouse_df["replicate"]==rep)].copy()
    if len(sub) < 20:
        print(f"{group} {rep}: Too few rows for PCA plot.")
        return
    Z = sub[["z_U_1_3h","z_log_U_over_C"]].values
    pca = PCA(n_components=2, random_state=0)
    PC = pca.fit_transform(Z)
    sub["PC1"], sub["PC2"] = PC[:,0], PC[:,1]

    plt.figure(figsize=(7,5))
    m = sub["is_event_or_next"].astype(bool)
    plt.scatter(sub.loc[~m,"PC1"], sub.loc[~m,"PC2"], s=18, alpha=0.55, label="normal")
    plt.scatter(sub.loc[m,"PC1"], sub.loc[m,"PC2"], s=20, alpha=0.85, label="event±1")
    plt.title(f"{group} {rep}: PCA on z-features (QC-kept)")
    plt.xlabel("PC1"); plt.ylabel("PC2"); plt.legend()
    plt.tight_layout(); plt.show()

    plt.figure(figsize=(7,5))
    sc = plt.scatter(sub["PC1"], sub["PC2"], c=sub["p_lowU"], s=18, alpha=0.85)
    plt.title(f"{group} {rep}: PCA colored by p_lowU")
    plt.xlabel("PC1"); plt.ylabel("PC2")
    plt.colorbar(sc, label="p_lowU")
    plt.tight_layout(); plt.show()

if len(mouse_df) > 0:
    for group in GROUPS.keys():
        for rep in GROUPS[group].keys():
            plot_rep_pca(mouse_df, group, rep)

---
## 7) Cage-by-cage and Replicate summaries + Rep1 vs Rep2 comparison

In [ ]:
mouse_summary = build_mouse_summary_table(mouse_df, lag=LAG_DAYS, n_perm=N_PERM)
cage_summary = agg_level(mouse_summary, ["group","replicate","cage_id"])
rep_summary  = agg_level(mouse_summary, ["group","replicate"])
overall_summary = agg_level(mouse_summary, ["group"])

display(cage_summary.sort_values(["group","replicate","cage_id"]))
display(rep_summary.sort_values(["group","replicate"]))
display(overall_summary.sort_values(["group"]))

In [ ]:
for g in GROUPS.keys():
    t = rep_vs_rep_tests(mouse_summary, g)
    if len(t) > 0:
        display(t)

---
## 8) Event sensitivity: recompute mean autocorr after masking events

In [ ]:
masked_event = recompute_autocorr_masking_events(mouse_df, mask_mode="event_only")
masked_evt1  = recompute_autocorr_masking_events(mouse_df, mask_mode="event_or_next")

m1 = masked_event.groupby(["group","replicate"]).agg(mean_autocorr_masked=("obs_autocorr_lag_masked","mean")).reset_index()
m2 = masked_evt1.groupby(["group","replicate"]).agg(mean_autocorr_masked=("obs_autocorr_lag_masked","mean")).reset_index()
event_sensitivity = m1.merge(m2, on=["group","replicate"], suffixes=("_event_only","_event_or_next"))
display(event_sensitivity)

---
## 9) FINAL SUMMARY CELL (copy/paste output)

Copy the full output of this cell into chat for interpretation.

In [ ]:
print("="*120)
print("RUN SUMMARY")
print("="*120)

if len(feat) == 0:
    print("No data loaded. Check cages/dates/path.")
else:
    base = feat.drop_duplicates(["group","replicate","cage_id","animal_id","day"]).copy()
    qc_rate = base.groupby(["group","replicate"]).agg(
        n_mouse_days=("qc_keep","size"),
        qc_keep_rate=("qc_keep","mean"),
        n_mice=("animal_id","nunique"),
        n_cages=("cage_id","nunique"),
    ).reset_index()
    print("\nQC retention (mouse-days):")
    print(qc_rate.round(4).to_string(index=False))

print("\n" + "="*120)
print("REPLICATE SUMMARY (aggregated across mice)")
print("="*120)
if len(rep_summary) > 0:
    cols = ["group","replicate","n_mice","mean_autocorr","median_autocorr","frac_mice_mc_p_lt_0_05",
            "mean_event_minus_non","frac_mice_event_p_lt_0_05","frac_high_p"]
    print(rep_summary[cols].round(4).to_string(index=False))
else:
    print("No replicate summary available.")

print("\n" + "="*120)
print("CAGE SUMMARY (aggregated across mice)")
print("="*120)
if len(cage_summary) > 0:
    cols = ["group","replicate","cage_id","n_mice","mean_autocorr","median_autocorr",
            "frac_mice_mc_p_lt_0_05","mean_event_minus_non","frac_mice_event_p_lt_0_05","frac_high_p"]
    print(cage_summary[cols].round(4).sort_values(["group","replicate","cage_id"]).to_string(index=False))
else:
    print("No cage summary available.")

print("\n" + "="*120)
print("TOP MICE BY CYCLICITY (lowest MC p-values)")
print("="*120)
if len(mouse_summary) > 0:
    top = mouse_summary.sort_values(["mc_pvalue","obs_autocorr_lag"], ascending=[True, False]).head(15)
    print(top[["group","replicate","cage_id","animal_id","n_days_qc","obs_autocorr_lag","mc_pvalue",
               "mean_p_event","mean_p_non_event","mwu_p_event_greater","frac_p_gt_0.9","frac_p_lt_0.1"]].round(4).to_string(index=False))
else:
    print("No per-mouse summary available.")

print("\n" + "="*120)
print("REP1 vs REP2 TESTS (mouse-level Mann–Whitney U)")
print("="*120)
if len(mouse_summary) > 0:
    for g in sorted(mouse_summary["group"].unique()):
        t = rep_vs_rep_tests(mouse_summary, g)
        if len(t) == 0:
            print(f"{g}: not enough reps or data")
        else:
            print(f"\nGroup={g}")
            print(t.round(4).to_string(index=False))

print("\n" + "="*120)
print("EVENT SENSITIVITY (mean autocorr@lag after masking events)")
print("="*120)
if 'event_sensitivity' in globals() and len(event_sensitivity) > 0:
    print(event_sensitivity.round(4).to_string(index=False))
else:
    print("No event sensitivity computed.")

print("\nDone.")